# Carta Azul: pipeline validation

Run this notebook in the authorized Databricks workspace after a successful historical update. It validates the published tables and executes the existing 28 synthetic transformation assertions. It does not create, modify, run or reset a Job or pipeline, and it does not write source files or persistent tables. The synthetic harness uses temporary views and removes them afterwards.

This is a curated, reproducible companion to the original `03_carta_azul_deploy_verify` notebook. Deployment payloads, authentication details and historical repair commands are deliberately excluded. Outputs are cleared in Git; run outputs belong in the private workspace. Accepted processing runs: `438408340438199` and `939543104005830` (2026-10-27).

Prerequisites: access to the project tables, the uploaded source files and test harness; a native Databricks Python/Spark notebook session; the bundled Databricks SDK for the optional run-status check. Start with the operational README. Keep this notebook outside the pipeline's `libraries` list.


In [ ]:
from datetime import date, timedelta
import importlib
import json
from pathlib import Path
import re
import sys
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "applebees"
WORKSPACE_PROJECT_ROOT = "/Workspace/Users/a01352461@tec.mx/CartaAzul"
EXPECTED_START = date(2026, 4, 28)
EXPECTED_END = date(2026, 8, 20)
EXPECTED_SOURCE_FILES = 115
PIPELINE_ID = "869fc958-69e2-4f3b-ae58-616b27696816"
ACCEPTED_RUN_IDS = [438408340438199, 939543104005830]
CHECK_ACCEPTED_RUN_STATUS = True

assert all(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", x) for x in (CATALOG, SCHEMA))
def project_table(name):
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"


## 1. Confirm the processing boundary

Both historical acceptance runs must have succeeded. Inspect the pipeline before consuming multiple tables: Lakeflow does not provide a global transaction across all outputs. The following SDK calls are read-only and use native notebook authentication. No credentials are stored in this repository.


In [ ]:
if CHECK_ACCEPTED_RUN_STATUS:
    from databricks.sdk import WorkspaceClient
    validation_client = WorkspaceClient()
    current_pipeline = validation_client.pipelines.get(pipeline_id=PIPELINE_ID)
    assert current_pipeline.state.value == "IDLE", "Wait until the project pipeline is idle before validating its tables."
    for run_id in ACCEPTED_RUN_IDS:
        accepted_run = validation_client.jobs.get_run(run_id=run_id)
        assert accepted_run.state.result_state.value == "SUCCESS", f"Accepted run {run_id} did not succeed."
    print(json.dumps({"accepted_job_runs": len(ACCEPTED_RUN_IDS), "status": "SUCCESS", "pipeline_state": "IDLE"}))


## 2. Execute the existing synthetic regression tests

The harness executes the real Silver function bodies and Gold queries, adapting only notebook configuration access and dataset declarations. The deployed pipeline separately validates the declarative wrappers and expectations. Reload the schema after any source upload to avoid stale Python import caches. Tests have their own synthetic configuration; do not add unsupported custom Spark settings to a serverless notebook.


In [ ]:
PIPELINE_TRANSFORMATIONS_ROOT = str(Path(WORKSPACE_PROJECT_ROOT) / "src/transformations")
if PIPELINE_TRANSFORMATIONS_ROOT not in sys.path:
    sys.path.insert(0, PIPELINE_TRANSFORMATIONS_ROOT)
importlib.invalidate_caches()
import silver._schemas
importlib.reload(silver._schemas)
harness_path = Path(WORKSPACE_PROJECT_ROOT) / "resources/pipelines/test_pipeline_spark.py"
assert harness_path.is_file(), "Upload the unchanged test harness and source files before running this cell."
exec(compile(harness_path.read_text(), str(harness_path), "exec"), globals())
assert carta_azul_synthetic_test_results["status"] == "PASS"
assert carta_azul_synthetic_test_results["count"] == 28


## 3. Validate the complete historical snapshot

Expected counts describe the accepted closed dataset. A changed dataset or an intentionally restricted pilot should fail these checks; do not silently replace the baseline. Business-quality warnings remain reviewable even when structural checks pass.


In [ ]:
quality_rows = spark.table(project_table("gold_pipeline_quality")).take(2)
assert len(quality_rows) == 1, "The quality output must contain exactly one current-snapshot row."
quality = quality_rows[0].asDict(recursive=True)
expected_quality = {
    "source_record_count": 2142846,
    "canonical_ticket_count": 2142846,
    "source_file_count": EXPECTED_SOURCE_FILES,
    "partition_date_count": 115,
    "business_date_count": 115,
    "feature_ticket_count": 2142846,
    "distinct_feature_ticket_count": 2142846,
    "invalid_ticket_key_count": 0,
    "duplicate_ticket_key_count": 0,
    "payment_event_count": 2232092,
    "discount_event_count": 516690,
    "error_issue_row_count": 0,
    "exploratory_employee_ticket_candidate_count": 41475,
    "exploratory_review_case_count": 424,
    "exploratory_review_ticket_count": 424,
    "high_recorded_tip_case_count": 144,
    "high_discount_share_case_count": 280,
    "reconciliation_exception_count": 59,
    "item_discount_reconciliation_exception_count": 2,
    "tickets_with_unexpected_positive_raw_gratuity": 3,
    "snapshot_quality_status": "COMPLETE_WITH_ISSUES_TO_REVIEW",
    "rule_version": "v1_exploratory",
}
for field, expected in expected_quality.items():
    assert quality[field] == expected, f"Unexpected snapshot value for {field}: {quality[field]}"
assert quality["payment_event_count"] == quality["expected_payment_event_count"]
assert quality["discount_event_count"] == quality["expected_discount_event_count"]
assert quality["first_partition_date"] == EXPECTED_START
assert quality["last_partition_date"] == EXPECTED_END
expected_dates = {EXPECTED_START + timedelta(days=i) for i in range((EXPECTED_END - EXPECTED_START).days + 1)}
actual_dates = {r.partition_date for r in spark.table(project_table("silver_checks")).select("partition_date").distinct().collect()}
assert actual_dates == expected_dates, "The historical date set contains a gap or an unexpected partition."
print(json.dumps({"snapshot": "PASS", "source_files": quality["source_file_count"], "tickets": quality["canonical_ticket_count"], "quality_status": quality["snapshot_quality_status"]}))


## 4. Check row counts and canonical/event uniqueness

The two original full runs also matched aggregate content fingerprints for all 11 tables. These lighter checks verify the current row counts and unique keys; they do not perform a fresh two-run fingerprint comparison. The original aggregate signatures are a practical stability check, not a formal row-by-row proof.


In [ ]:
expected_rows = {
    "silver_checks": 2142846,
    "silver_payments": 2232092,
    "silver_discounts": 516690,
    "silver_employees": 156975,
    "silver_locations": 185,
    "silver_revenue_centers": 23,
    "silver_quality_issues": 30232,
    "gold_check_features": 2142846,
    "gold_discount_daily": 155887,
    "gold_review_cases": 16033,
    "gold_pipeline_quality": 1,
}
primary_keys = {
    "silver_checks": "ticket_key", "gold_check_features": "ticket_key",
    "silver_payments": "payment_event_key", "silver_discounts": "discount_event_key",
    "silver_employees": "employee_id", "silver_locations": "location_id",
    "silver_revenue_centers": "revenue_center_id", "gold_review_cases": "case_key",
}
observed_rows = {}
for name, expected_count in expected_rows.items():
    frame = spark.table(project_table(name))
    if name in primary_keys:
        row = frame.agg(F.count("*").alias("rows"), F.countDistinct(primary_keys[name]).alias("keys")).first()
        assert row.rows == row.keys, f"Null or duplicated primary keys in {name}"
        count = row.rows
    else:
        count = frame.count()
    assert count == expected_count, f"Unexpected row count in {name}: {count}"
    observed_rows[name] = count
print(json.dumps({"tables": len(observed_rows), "row_counts": observed_rows, "status": "PASS"}))


## 5. Inspect aggregate quality issues

Rules overlap. `issue_rows`, distinct tickets, source rows and review cases are different units. `ManagerId = 0` is informational context and does not establish missing authorization. A successful pipeline with source warnings is expected for this historical snapshot.


In [ ]:
issue_summary = spark.sql(f"""
    SELECT issue_code, severity, COUNT(*) AS issue_rows,
           COUNT(DISTINCT ticket_key) AS affected_tickets
    FROM {project_table('silver_quality_issues')}
    GROUP BY issue_code, severity
    ORDER BY severity, issue_code
""")
display(issue_summary)
print("PIPELINE_VALIDATION_PASS")
